#Configuration

In [0]:
STORAGE_ACCOUNT = "cmsmedicaredatastorage"
CONTAINER = "cms-medicare-raw"
SOURCE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/provider_utilization/"
CHECKPOINT_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/bronze_provider/"
TARGET_TABLE = "cms_medicare_databricks_pipeline.bronze.provider_utilization_raw"

#Auto Loader

In [0]:

df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
    .load(SOURCE_PATH)
)

#Add metadata columns
For debugging and auditing

In [0]:
from pyspark.sql import functions as F

df_with_metadata = df_raw.select(
    "*",
    F.current_timestamp().alias("ingestion_timestamp"),
    F.input_file_name().alias("source_file")
)

#Write to Delta Table

In [0]:
(
    df_with_metadata
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

#Verify

In [0]:
count = spark.read.table(TARGET_TABLE).count()
print(f"Total rows ingested: {count:,}")

display(spark.read.table(TARGET_TABLE).limit(5))
spark.table(TARGET_TABLE).printSchema()